# FIFA World Cup 2026 — Linear Regression 2.1
 # 01. Data Preparation: verified pre-match predictors

**Objective:** Predict home-team goal_difference using exactly eight historical predictors. The fixture workbook supplies the 104 matches and outcomes; results.csv supplies dated international-match histories. For **every** fixture, match counts use the **preceding 365 calendar days**, while goals scored and win rates use the **up to ten most recent international matches on earlier calendar dates**. This avoids using the current match outcome to create its predictors.

**Inputs (keep alongside this notebook):** fifa-world-cup-2026-UTC.xlsx, results.csv.Outputs: Regression_2_1_Prepared_Data.csv and Regression_2_1_Historical_Audit.csv

## Step 1 — Load and inspect both datasets

In [19]:

import pandas as pd #importing pandas for data manipulation
import numpy as np #importing numpy for numerical operations
from pathlib import Path #importing Path from pathlib for handling file paths

pd.set_option("display.max_columns", None)  #display all columns in the output
fixture_path = "fifa-world-cup-2026-UTC.xlsx"  #path to the World Cup fixtures file
history_path = "results.csv"  #path to the historical international matches file

fixtures = pd.read_excel(fixture_path, sheet_name="Fixture")  #read the World Cup fixtures into a DataFrame
history = pd.read_csv(history_path)  #read the historical international matches into a DataFrame
print("World Cup fixtures:", fixtures.shape)  #print the shape of the World Cup fixtures DataFrame
print("Historical international matches:", history.shape)  #print the shape of the historical international matches DataFrame
display(fixtures.head(3))  #display the first 3 rows of the World Cup fixtures DataFrame
display(history.head(3))  #display the first 3 rows of the historical international matches DataFrame

World Cup fixtures: (104, 8)
Historical international matches: (49945, 9)


,Match Number,Round Number,Date,Location,Home Team,Away Team,Group,Result
0,1,1,46184.791667,Mexico City Stadium,Mexico,South Africa,Group A,2 - 0
1,2,1,46185.083333,Guadalajara Stadium,Korea Republic,Czechia,Group A,2 - 1
2,3,1,46185.791667,Toronto Stadium,Canada,Bosnia and Herzegovina,Group B,1 - 1


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False


In [20]:
print("Fixture columns:", fixtures.columns.tolist()) #print the list of columns in the World Cup fixtures DataFrame
print("History columns:", history.columns.tolist()) #print the list of columns in the historical international matches DataFrame
print("Missing fixture values:\n", fixtures.isna().sum()) #print the count of missing values in each column of the World Cup fixtures DataFrame
print("Missing historical values:\n", history.isna().sum()) #print the count of missing values in each column of the historical international matches DataFrame
print("Duplicate fixture match numbers:", fixtures["Match Number"].duplicated().sum()) #print the number of duplicate match numbers in the World Cup fixtures DataFrame
print("Duplicate historical rows:", history.duplicated().sum()) #print the number of duplicate rows in the historical international matches DataFrame

Fixture columns: ['Match Number', 'Round Number', 'Date', 'Location', 'Home Team', 'Away Team', 'Group', 'Result']
History columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
Missing fixture values:
 Match Number     0
Round Number     0
Date             0
Location         0
Home Team        0
Away Team        0
Group           32
Result           0
dtype: int64
Missing historical values:
 date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64
Duplicate fixture match numbers: 0
Duplicate historical rows: 0


## Step 2 — Clean dates, scores and team names

The fixture workbook contains Excel date-time values in UTC; the historical CSV contains calendar dates only. To prevent ambiguous same-day ordering, historical matches **on the fixture date are excluded**, even if they may have finished earlier. Historical dates must be strictly earlier than the World Cup match date.

In [21]:
# Convert fixture Excel serial dates into UTC datetimes, retaining the kickoff time
clean = fixtures.rename(columns={ #rename columns for consistency and clarity
    "Match Number": "match_id", "Round Number": "Round_Number",
    "Home Team": "Home_Team", "Away Team": "Away_Team",
    "Date": "date", "Result": "Result"
}).copy() #create a copy of the renamed DataFrame to avoid modifying the original fixtures DataFrame
clean["date"] = pd.to_datetime(clean["date"], unit="D", origin="1899-12-30", errors="coerce") if pd.api.types.is_numeric_dtype(clean["date"]) else pd.to_datetime(clean["date"], errors="coerce")  #convert fixture Excel serial dates into UTC datetimes, retaining the kickoff time

# Align different team-name spellings between the two datasets
aliases = {
    "Korea Republic": "South Korea", "Czechia": "Czech Republic",
    "USA": "United States", "Türkiye": "Turkey",
    "Côte d'Ivoire": "Ivory Coast", "Cabo Verde": "Cape Verde",
    "IR Iran": "Iran", "Congo DR": "DR Congo",
}
def standard_name(name):  #standardize team names using the aliases dictionary
    name = " ".join(str(name).replace("\xa0", " ").split()) #remove non-breaking spaces and extra whitespace from team names
    return aliases.get(name, name)  #return the standardized team name, or the original name if no alias is found

for col in ["Home_Team", "Away_Team"]:  #standardize team names in the fixtures DataFrame. This ensures consistency with the historical matches DataFrame.
    clean[col] = clean[col].map(standard_name) 
for col in ["home_team", "away_team"]:  #standardize team names in the historical matches DataFrame for consistency with the fixtures DataFrame
    history[col] = history[col].map(standard_name)
history["date"] = pd.to_datetime(history["date"], errors="coerce")  #convert historical match dates into UTC datetimes
for col in ["home_score", "away_score"]:  #convert historical match scores into numeric values
    history[col] = pd.to_numeric(history[col], errors="coerce")
history = history.dropna(subset=["date", "home_score", "away_score"]).drop_duplicates().copy()  #remove rows with missing dates or scores and drop duplicate rows

# Parse match scores; scores are used only to create the target, not the predictors
scores = clean["Result"].astype(str).str.extract(r"^\s*(\d+)\s*-\s*(\d+)\s*$") #Using a regular expression to extract home and away scores from the fixture results. The formula captures two groups of digits separated by a hyphen, ignoring leading and trailing whitespace.It then assigns the captured groups to separate columns in the resulting DataFrame.Then the extracted scores are used to calculate the goal difference for each fixture.
assert scores.notna().all().all(), "Unrecognised fixture result format" #ensure that all fixture results match the expected "home_score - away_score" format
clean["goal_difference"] = scores[0].astype(int) - scores[1].astype(int)  #calculate the goal difference for each fixture by subtracting the away score from the home score
clean["match_id"] = pd.to_numeric(clean["match_id"], errors="raise").astype(int) #convert match IDs to numeric values and ensure they are integers
assert clean["match_id"].is_unique and clean["date"].notna().all()  #ensure that match IDs are unique and all dates are valid
assert (clean["Home_Team"] != clean["Away_Team"]).all()  #ensure that no fixture has the same team playing against itself
print("Date range:", clean["date"].min(), "to", clean["date"].max())  #print the range of fixture dates
print("Fixture rows:", len(clean), "| unique matches:", clean["match_id"].nunique())  #print the number of fixture rows and unique match IDs

#The Date Range confirms the earliest and latest fixture dates in the cleaned DataFrame
#Fixture rows and unique matches provide an overview of the number of fixtures and ensure that match IDs are unique

Date range: 2026-06-11 19:00:00.000002931 to 2026-07-19 19:00:00.000002931
Fixture rows: 104 | unique matches: 104


## Step 3 — Define the eight predictors

For each team before each fixture: **prior matches** = total international matches played during the preceding 365 calendar days (not capped at ten); **average goals scored** = mean goals scored in up to ten most recent earlier-dated international matches; **rest days** = calendar days since the latest earlier-dated international match; **prior win rate** = wins divided by the number of matches in that last-ten window. Both home and away teams use identical definitions. Matches on the fixture date are excluded because the historical CSV has no kickoff times.

In [22]:
target = "goal_difference"  #define the target variable for the regression model
features = [  #define the list of feature columns for the regression model
    "Team_prior_matches", "Opponent_prior_matches",
    "Team_prior_goals_scored_avg", "Opponent_prior_goals_scored_avg",
    "Team_rest_days", "Opponent_rest_days",
    "Team_prior_win_rate", "Opponent_prior_win_rate",
]
assert len(features) == 8 and len(set(features)) == 8  #ensure that there are exactly 8 features and all are unique
for i, col in enumerate(features, 1): # Loop through the features and print their names with indices
    print(f"X{i}: {col}")  #print the feature names with their corresponding indices

X1: Team_prior_matches
X2: Opponent_prior_matches
X3: Team_prior_goals_scored_avg
X4: Opponent_prior_goals_scored_avg
X5: Team_rest_days
X6: Opponent_rest_days
X7: Team_prior_win_rate
X8: Opponent_prior_win_rate


## Step 4 — Build dated international match histories

Each international match is recorded for both teams, showing the goals each team scored and whether they won. For every World Cup fixture, the previous matches are selected separately to calculate the historical statistics. This also includes earlier World Cup matches if their results are already available in the international results dataset.

In [23]:
home_records = history[["date", "home_team", "home_score", "away_score"]].rename(  #create a DataFrame for home team records with standardized column names
    columns={"home_team": "team", "home_score": "goals_for", "away_score": "goals_against"})
away_records = history[["away_team", "away_score", "home_score"]].rename(  #create a DataFrame for away team records with standardized column names
    columns={"away_team": "team", "away_score": "goals_for", "home_score": "goals_against"})
team_history = pd.concat([home_records, away_records], ignore_index=True)  #combine home and away records into a single DataFrame for each team's match history
team_history["win"] = (team_history["goals_for"] > team_history["goals_against"]).astype(int)  #create a binary column indicating whether the team won the match
team_history = team_history.sort_values(["team", "date"], kind="stable").reset_index(drop=True)  #sort the team history by team and date for chronological order

teams = set(clean["Home_Team"]) | set(clean["Away_Team"]) #get the set of all teams participating in the World Cup fixtures
missing_teams = sorted(teams - set(team_history["team"]))  #identify teams that are in the World Cup fixtures but have no historical match records
print("World Cup teams:", len(teams))  #print the total number of World Cup teams
print("Teams without international-match records:", missing_teams)  #print the list of teams without historical match records
assert not missing_teams, "Resolve unmatched team names before modelling"  #ensure that all World Cup teams have corresponding historical match records

World Cup teams: 48
Teams without international-match records: []


In [24]:
# Compute features using only international matches before the fixture date
history_by_team = {name: group for name, group in team_history.groupby("team")}

def prior_statistics(team, kickoff): #Define a function that computes prior statistics for a given team before a specific fixture date
    # All matches must be on earlier calendar dates (no same-day leakage).
    earlier = history_by_team[team]  #get all historical matches for the given team
    earlier = earlier.loc[earlier["date"] < kickoff.normalize()] #filter out matches that occur on or after the fixture date

    # Match counts measure international activity during the preceding 365 days.
    start_date = kickoff.normalize() - pd.Timedelta(days=365)  #compute the start date for the 365-day window
    matches_365 = int((earlier["date"] >= start_date).sum()) #count the number of matches in the preceding 365 days

    # Other performance measures retain the original last-ten-match definition.
    previous = earlier.tail(10)  #get the last ten matches for the given team
    if previous.empty: # conditional statement for handling cases with no previous matches
        return {"matches": matches_365, "matches_used": 0, #return default values when there are no previous matches
                "goals_avg": np.nan, "rest_days": np.nan, #default values for goals average and rest days
                "win_rate": np.nan, "latest_date": pd.NaT, "earliest_date": pd.NaT} #default values for win rate and history dates
    return {
        "matches": matches_365,     #number of matches in the preceding 365 days
        "matches_used": len(previous),  #number of matches used for computing other performance measures
        "goals_avg": previous["goals_for"].mean(),  #average goals scored in the last ten matches
        "rest_days": (kickoff.normalize() - previous["date"].max()).days,  #number of rest days since the last match
        "win_rate": previous["win"].mean(),  #win rate in the last ten matches
        "latest_date": previous["date"].max(),  #date of the most recent match in the last ten matches
        "earliest_date": previous["date"].min(),  #date of the earliest match in the last ten matches
    }

rows = [] #Initialize list to store calculated predictor rows
audit_rows = [] #Initialize list to store audit information for each fixture
for fixture in clean.itertuples(index=False):                  #Iterate over each fixture in the cleaned dataset
    home = prior_statistics(fixture.Home_Team, fixture.date)   #Calculate prior statistics for the home team
    away = prior_statistics(fixture.Away_Team, fixture.date)   #Calculate prior statistics for the away team
    rows.append({                                 #Append calculated predictor values for the current fixture
        "match_id": fixture.match_id,             #Unique identifier for the current fixture
        "Team_prior_matches": home["matches"],  #Number of matches played by the home team in the preceding 365 days
        "Opponent_prior_matches": away["matches"],  #Number of matches played by the away team in the preceding 365 days
        "Team_prior_goals_scored_avg": home["goals_avg"],  #Average goals scored by the home team in the last ten matches
        "Opponent_prior_goals_scored_avg": away["goals_avg"],  #Average goals scored by the away team in the last ten matches
        "Team_rest_days": home["rest_days"],  #Number of rest days for the home team since their last match
        "Opponent_rest_days": away["rest_days"],  #Number of rest days for the away team since their last match
        "Team_prior_win_rate": home["win_rate"],  #Win rate of the home team in the last ten matches
        "Opponent_prior_win_rate": away["win_rate"],  #Win rate of the away team in the last ten matches
    })
    for role, name, stats in [("Home", fixture.Home_Team, home),         #Iterate over both home and away teams to store audit information
                              ("Away", fixture.Away_Team, away)]:        #Loop over both teams to create audit rows
        audit_rows.append({"match_id": fixture.match_id, "fixture_date": fixture.date, #Append audit information for the current team
                           "role": role, "team": name,              #Role and name of the current team
                           "matches_365_days": stats["matches"],  #Number of matches played by the team in the preceding 365 days
                           "matches_used": stats["matches_used"],  #Number of matches used to calculate the statistics
                           "earliest_history_date": stats["earliest_date"],  #Earliest date of the team's match history considered
                           "latest_history_date": stats["latest_date"],  #Latest date of the team's match history considered
                           "average_goals": stats["goals_avg"],  #Average goals scored by the team in the last ten matches
                           "rest_days": stats["rest_days"], "win_rate": stats["win_rate"]})  #Number of rest days since the last match and win rate of the team in the last ten matches

calculated = pd.DataFrame(rows)   #Create a DataFrame from the calculated rows containing match statistics
audit = pd.DataFrame(audit_rows)   #Create a DataFrame from the audit rows containing detailed team statistics
clean = clean.merge(calculated, on="match_id", validate="one_to_one")   #Merge the calculated statistics with the clean DataFrame
print("Calculated predictor rows:", len(calculated))   #Print the number of calculated predictor rows
display(clean[["match_id", "Home_Team", "Away_Team"] + features].head(10))   #Display the first 10 rows of the clean DataFrame with selected features

Calculated predictor rows: 104


,match_id,Home_Team,Away_Team,Team_prior_matches,Opponent_prior_matches,Team_prior_goals_scored_avg,Opponent_prior_goals_scored_avg,Team_rest_days,Opponent_rest_days,Team_prior_win_rate,Opponent_prior_win_rate
0,1,Mexico,South Africa,16,10,1.5,1.5,7,13,0.4,0.4
1,2,South Korea,Czech Republic,7,7,1.9,2.0,1,12,0.7,0.6
2,3,Canada,Bosnia and Herzegovina,11,4,1.0,1.1,7,14,0.3,0.3
3,4,United States,Paraguay,17,2,2.2,1.4,1,8,0.6,0.7
4,8,Qatar,Switzerland,10,5,0.9,2.7,7,13,0.1,0.6
5,7,Brazil,Morocco,7,21,2.2,2.2,7,6,0.7,0.7
6,5,Haiti,Scotland,9,5,1.0,1.7,1,15,0.3,0.5
7,6,Australia,Turkey,6,5,1.9,1.8,1,13,0.7,0.7
8,10,Germany,Curaçao,5,6,3.1,2.5,14,8,0.7,0.7
9,11,Netherlands,Japan,7,5,2.8,2.7,6,14,0.6,0.7


## Step 5 — Verify historical timing and values

Check that last-ten-match performance measures use only earlier-dated records. Independently verify that each 365-day match count equals the number of source matches in the interval **[fixture date − 365 days, fixture date)**.

In [25]:
# Check every historical date is strictly earlier than its World Cup fixture date
has_history = audit["matches_used"] > 0   #Boolean mask indicating which audit rows have historical matches used
assert (audit.loc[has_history, "latest_history_date"].dt.normalize() < #Ensure that the latest historical match date is earlier than the fixture date for rows with history
        audit.loc[has_history, "fixture_date"].dt.normalize()).all() #Check that all historical match dates are earlier than the fixture date
assert audit["matches_used"].between(0, 10).all()  #Ensure that the number of matches used is between 0 and 10
assert (audit["matches_365_days"] >= 0).all()  #Ensure that the number of matches in the last 365 days is non-negative
assert (audit["matches_365_days"] == [   #Ensure that the number of matches in the last 365 days matches the calculated value based on the team's match history
    int(((history_by_team[row.team]["date"] >= row.fixture_date.normalize() - pd.Timedelta(days=365)) &
         (history_by_team[row.team]["date"] < row.fixture_date.normalize())).sum())
    for row in audit.itertuples(index=False)
]).all()  #Ensure that the number of matches in the last 365 days matches the calculated value
assert (audit.loc[has_history, "rest_days"] >= 1).all()  #Ensure that the number of rest days since the last match is at least 1 for rows with history
assert audit.loc[has_history, "win_rate"].between(0, 1).all()  #Ensure that the win rate is between 0 and 1 for rows with history
assert (audit.loc[has_history, "average_goals"] >= 0).all()  #Ensure that the average goals scored is non-negative for rows with history
print("Team-fixture histories with fewer than ten earlier matches:", (audit["matches_used"] < 10).sum())   #Print the number of team-fixture histories with fewer than ten earlier matches
print("365-day home match-count unique values:", clean["Team_prior_matches"].nunique())   #Print the number of unique values for the home team's match count in the last 365 days
print("365-day away match-count unique values:", clean["Opponent_prior_matches"].nunique())   #Print the number of unique values for the away team's match count in the last 365 days
print("Missing predictor values:\n", clean[features].isna().sum())   #Print the number of missing values for each predictor feature
print("Number of audit rows:", len(audit))   #Print the total number of audit rows
display(clean[features + [target]].describe().T)   #Display descriptive statistics for the selected features and target variable
assert clean[features].notna().all().all(), "Some teams lack history; review missing values before modelling"   #Ensure that there are no missing values for the predictor features

Team-fixture histories with fewer than ten earlier matches: 0
365-day home match-count unique values: 15
365-day away match-count unique values: 17
Missing predictor values:
 Team_prior_matches                 0
Opponent_prior_matches             0
Team_prior_goals_scored_avg        0
Opponent_prior_goals_scored_avg    0
Team_rest_days                     0
Opponent_rest_days                 0
Team_prior_win_rate                0
Opponent_prior_win_rate            0
dtype: int64
Number of audit rows: 208


,count,mean,std,min,25%,50%,75%,max
Team_prior_matches,104.0,8.067308,3.484402,2.0,6.000,7.0,9.000,21.0
Opponent_prior_matches,104.0,8.192308,4.126817,2.0,6.000,7.0,10.000,22.0
Team_prior_goals_scored_avg,104.0,2.136538,0.695067,0.9,1.575,2.2,2.600,3.7
Opponent_prior_goals_scored_avg,104.0,1.981731,0.664899,0.8,1.400,1.9,2.425,3.9
Team_rest_days,104.0,12.740385,35.560968,1.0,1.000,6.0,9.000,254.0
Opponent_rest_days,104.0,16.576923,32.107433,1.0,6.000,8.5,14.000,248.0
Team_prior_win_rate,104.0,0.617308,0.162769,0.1,0.500,0.6,0.700,1.0
Opponent_prior_win_rate,104.0,0.614423,0.165123,0.1,0.500,0.6,0.700,1.0
goal_difference,104.0,0.480769,2.024034,-4.0,-1.000,0.0,2.000,6.0


## Step 6 — Verify World Cup results against the historical CSV

Cross-check where a corresponding World Cup match exists in the historical dataset. The historical CSV records only dates, not kickoff times, so this comparison validates results but does not replace the conservative earlier-date cutoff.

In [26]:
wc_history = history.loc[history["tournament"].eq("FIFA World Cup")].copy()  #Filter the historical match data to include only FIFA World Cup matches and create a separate DataFrame for them
wc_history["day"] = wc_history["date"].dt.normalize()  #Create a normalized day column for the World Cup historical matches
check = clean[["match_id", "date", "Home_Team", "Away_Team", "goal_difference"]].copy()  #Create a copy of the clean DataFrame with selected columns for matching
check["day"] = check["date"].dt.normalize()  #Create a normalized day column for the clean DataFrame to match with World Cup historical matches
matched = check.merge(wc_history, left_on=["day", "Home_Team", "Away_Team"],
                      right_on=["day", "home_team", "away_team"], how="left", indicator=True)  #Merge the clean DataFrame with World Cup historical matches to find corresponding records
matched["history_difference"] = matched["home_score"] - matched["away_score"]  #Calculate the goal difference from the historical match scores
found = matched["_merge"].eq("both")  #Boolean mask indicating which matches were found in both the clean DataFrame and historical records
conflicts = matched.loc[found & matched["history_difference"].ne(matched["goal_difference"])]  #Identify matches where the goal difference conflicts between the clean DataFrame and historical records
print("World Cup fixtures matched to international-results records:", found.sum(), "of", len(clean))  #Print the number of World Cup fixtures successfully matched to historical records
print("Score discrepancies among matched fixtures:", len(conflicts))  #Print the number of score discrepancies among the matched fixtures
if len(conflicts):
    display(conflicts[["match_id", "Home_Team", "Away_Team", "goal_difference", "history_difference"]])  #Display the conflicting matches for review
assert len(conflicts) == 0, "Resolve score discrepancies between sources"  #Ensure that there are no score discrepancies between the clean DataFrame and historical records

World Cup fixtures matched to international-results records: 67 of 104
Score discrepancies among matched fixtures: 0


## Step 7 — Export the final prepared dataset and audit trail

Keep match identifiers and dates for chronological splitting, but do not use them or current-match scores as explanatory variables. Export a separate audit file for methodological traceability.

In [27]:
id_columns = ["match_id", "date", "Round_Number", "Home_Team", "Away_Team"]   #List of identifier columns for the dataset
final_columns = id_columns + features + [target]   #Combine identifier columns, feature columns, and target column to form the final set of columns
df_prepared = clean[final_columns].sort_values(["date", "match_id"], kind="stable").reset_index(drop=True)   #Create the prepared DataFrame by selecting the final columns and sorting by date and match ID
assert len(df_prepared) == 104 and df_prepared["match_id"].is_unique   #Ensure that the prepared DataFrame has the expected number of rows and unique match IDs
assert df_prepared[features + [target]].notna().all().all()   #Ensure that there are no missing values in the feature and target columns

output_file = Path("Regression_2_1_Prepared_Data.csv")   #Path to save the prepared dataset
audit_file = Path("Regression_2_1_Historical_Audit.csv")   #Path to save the historical audit dataset
df_prepared.to_csv(output_file, index=False)   #Save the prepared dataset to the specified CSV file
audit.to_csv(audit_file, index=False)   #Save the historical audit dataset to the specified CSV file
print("Prepared data saved:", output_file.resolve())   #Print the path to the saved prepared dataset
print("Historical audit saved:", audit_file.resolve())   #Print the path to the saved historical audit dataset
print("Final dataset shape:", df_prepared.shape)   #Print the shape of the final prepared dataset
display(df_prepared.head(10))   #Display the first 10 rows of the prepared dataset

Prepared data saved: C:\Users\jelag\OneDrive\Documents\MSC SPATIAL SCIENCE\FOUNDATIONS OF DATA SCIENCE\Group4_HIT140_LinearRegression2.1\Regression_2_1_Prepared_Data.csv
Historical audit saved: C:\Users\jelag\OneDrive\Documents\MSC SPATIAL SCIENCE\FOUNDATIONS OF DATA SCIENCE\Group4_HIT140_LinearRegression2.1\Regression_2_1_Historical_Audit.csv
Final dataset shape: (104, 14)


,match_id,date,Round_Number,Home_Team,Away_Team,Team_prior_matches,Opponent_prior_matches,Team_prior_goals_scored_avg,Opponent_prior_goals_scored_avg,Team_rest_days,Opponent_rest_days,Team_prior_win_rate,Opponent_prior_win_rate,goal_difference
0,1,2026-06-11 19:00:00.000002931,1,Mexico,South Africa,16,10,1.5,1.5,7,13,0.4,0.4,2
1,2,2026-06-12 01:59:59.999997068,1,South Korea,Czech Republic,7,7,1.9,2.0,1,12,0.7,0.6,1
2,3,2026-06-12 19:00:00.000002931,1,Canada,Bosnia and Herzegovina,11,4,1.0,1.1,7,14,0.3,0.3,0
3,4,2026-06-13 01:00:00.000002931,1,United States,Paraguay,17,2,2.2,1.4,1,8,0.6,0.7,3
4,8,2026-06-13 19:00:00.000002931,1,Qatar,Switzerland,10,5,0.9,2.7,7,13,0.1,0.6,0
5,7,2026-06-13 22:00:00.000002931,1,Brazil,Morocco,7,21,2.2,2.2,7,6,0.7,0.7,0
6,5,2026-06-14 01:00:00.000002931,1,Haiti,Scotland,9,5,1.0,1.7,1,15,0.3,0.5,-1
7,6,2026-06-14 04:00:00.000002931,1,Australia,Turkey,6,5,1.9,1.8,1,13,0.7,0.7,2
8,10,2026-06-14 16:59:59.999997068,1,Germany,Curaçao,5,6,3.1,2.5,14,8,0.7,0.7,6
9,11,2026-06-14 19:59:59.999997068,1,Netherlands,Japan,7,5,2.8,2.7,6,14,0.6,0.7,0


## Final Data Preparation Summary

Using the two datasets, we calculated eight historical predictors for all 104 World Cup matches. For Team_prior_matches and Opponent_prior_matches, we counted the number of international matches each team played in the 365 days before each fixture. To calculate average goals scored and win rate, we used each team's last 10 international matches, or fewer if 10 were not available. Rest days were calculated from the team's most recent match. we made sure that only matches played before each World Cup fixture were used in the calculations.

**Limitations:** The historical dataset only provides match dates and not the exact kickoff times. Because of this, we excluded matches played on the same date to avoid using results that might not have been available before the fixture. Rest days were calculated using calendar days rather than exact hours. We chose to use the previous 365 days for match counts and the last 10 matches for average goals and win rate. The audit CSV includes matches_365_days and matches_used to show the difference between these calculations. The updated prepared dataset will be used for the EDA and regression modelling stages.